In [0]:

CREATE OR REPLACE TEMP VIEW bronze_members AS
SELECT *
FROM delta.`abfss://bronze@incubytedatalake01.dfs.core.windows.net/member_data/`;

In [0]:
--Verify the reocrds
SELECT *
FROM bronze_members;

Member_Name,Member_ID,Enrollment_Date,Last_Flight_Date,Tier_Code,Agent_Name,State,Country,Post_Code,DOB,Is_Active,Member_Type,Source_Country,Ingested_At
Mike,1,2022-05-11 00:00:00,2022-08-01 00:00:00,PLT,null,null,null,null,NULL,null,null,AUS,2026-11-21T07:18:39.239Z
Jonnathan,2,2021-13-13,2022-01-05 00:00:00,GLD,null,null,null,null,1997-12-13 00:00:00,null,null,AUS,2026-11-21T07:18:39.239Z
Cristina,3,2022-03-12 00:00:00,2022-03-20 00:00:00,SLV,null,null,null,null,1998-03-12 00:00:00,null,null,AUS,2026-11-21T07:18:39.239Z
Vikas,1,2022-01-01 00:00:00,2022-06-15 00:00:00,SLV,null,null,null,null,1998-12-01 00:00:00,null,I,IND,2026-11-21T07:18:39.239Z
Rahul,2,2022-03-05 00:00:00,2022-03-10 00:00:00,GLD,null,null,null,null,1982-08-13 00:00:00,null,C,IND,2026-11-21T07:18:39.239Z
Sameer,3,2022-02-20 00:00:00,2022-02-25 00:00:00,GLD,null,null,null,null,1952-08-13 00:00:00,null,I,IND,2026-11-21T07:18:39.239Z
Sam,1,6152022,8202022,PLT,null,null,null,null,null,null,null,USA,2026-11-21T07:18:39.239Z
John,2,1052022,1152022,SLV,null,null,null,null,null,null,null,USA,2026-11-21T07:18:39.239Z
Mike,3,12282021,12302021,GLD,null,null,null,null,null,null,null,USA,2026-11-21T07:18:39.239Z


In [0]:
--Date values
SELECT DISTINCT
    Enrollment_Date,
    Last_Flight_Date,
    DOB
FROM bronze_members
ORDER BY Enrollment_Date;

Enrollment_Date,Last_Flight_Date,DOB
1052022,1152022,null
12282021,12302021,null
2021-13-13,2022-01-05 00:00:00,1997-12-13 00:00:00
2022-01-01 00:00:00,2022-06-15 00:00:00,1998-12-01 00:00:00
2022-02-20 00:00:00,2022-02-25 00:00:00,1952-08-13 00:00:00
2022-03-05 00:00:00,2022-03-10 00:00:00,1982-08-13 00:00:00
2022-03-12 00:00:00,2022-03-20 00:00:00,1998-03-12 00:00:00
2022-05-11 00:00:00,2022-08-01 00:00:00,NULL
6152022,8202022,null


In [0]:
--Duplicate member ids
SELECT
    Member_ID,
    COUNT(*) AS record_count
FROM bronze_members
GROUP BY Member_ID
HAVING COUNT(*) > 1;

Member_ID,record_count
1,3
2,3
3,3


In [0]:
--Duplicate Id by country
SELECT
    Member_ID,
    Source_Country,
    COUNT(*) AS record_count,
    COLLECT_LIST(Member_Name) AS member_names
FROM bronze_members
GROUP BY Member_ID, Source_Country
HAVING COUNT(*) > 1;

Member_ID,Source_Country,record_count,member_names


In [0]:

--Cross-Country Member ID Collision Analysis
-- Identify Member IDs appearing across multiple source countries.
-- Investigate whether these represent country moves or ID collisions.
SELECT
    Member_ID,
    COUNT(DISTINCT Source_Country) AS country_count,
    COLLECT_SET(Source_Country) AS countries,
    COLLECT_SET(Member_Name) AS member_names
FROM bronze_members
GROUP BY Member_ID
HAVING COUNT(DISTINCT Source_Country) > 1;

Member_ID,country_count,countries,member_names
1,3,"List(AUS, IND, USA)","List(Mike, Vikas, Sam)"
2,3,"List(AUS, IND, USA)","List(Jonnathan, Rahul, John)"
3,3,"List(AUS, IND, USA)","List(Cristina, Sameer, Mike)"


In [0]:
--Inspect date formats in Bronze data

SELECT
    Member_ID,
    Member_Name,
    Source_Country,
    Enrollment_Date,
    Last_Flight_Date,
    DOB
FROM bronze_members
ORDER BY Member_ID, Source_Country;

Member_ID,Member_Name,Source_Country,Enrollment_Date,Last_Flight_Date,DOB
1,Mike,AUS,2022-05-11 00:00:00,2022-08-01 00:00:00,NULL
1,Vikas,IND,2022-01-01 00:00:00,2022-06-15 00:00:00,1998-12-01 00:00:00
1,Sam,USA,6152022,8202022,null
2,Jonnathan,AUS,2021-13-13,2022-01-05 00:00:00,1997-12-13 00:00:00
2,Rahul,IND,2022-03-05 00:00:00,2022-03-10 00:00:00,1982-08-13 00:00:00
2,John,USA,1052022,1152022,null
3,Cristina,AUS,2022-03-12 00:00:00,2022-03-20 00:00:00,1998-03-12 00:00:00
3,Sameer,IND,2022-02-20 00:00:00,2022-02-25 00:00:00,1952-08-13 00:00:00
3,Mike,USA,12282021,12302021,null


In [0]:
--Date Standardisation
-- Step 6.1: Standardize date columns

CREATE OR REPLACE TEMP VIEW member_dates AS

SELECT
    *,

    COALESCE(
        TRY_TO_TIMESTAMP(Enrollment_Date, 'yyyy-MM-dd HH:mm:ss'),
        TRY_TO_TIMESTAMP(Enrollment_Date, 'Mddyyyy'),
        TRY_TO_TIMESTAMP(Enrollment_Date, 'MMddyyyy')
    ) AS Enrollment_Date_Standard,

    COALESCE(
        TRY_TO_TIMESTAMP(Last_Flight_Date, 'yyyy-MM-dd HH:mm:ss'),
        TRY_TO_TIMESTAMP(Last_Flight_Date, 'Mddyyyy'),
        TRY_TO_TIMESTAMP(Last_Flight_Date, 'MMddyyyy')
    ) AS Last_Flight_Date_Standard,

    TRY_TO_TIMESTAMP(DOB, 'yyyy-MM-dd HH:mm:ss') AS DOB_Standard

FROM bronze_members;

In [0]:
SELECT
    Member_ID,
    Member_Name,
    Source_Country,
    Enrollment_Date,
    Enrollment_Date_Standard,
    Last_Flight_Date,
    Last_Flight_Date_Standard,
    DOB_Standard
FROM member_dates
ORDER BY Member_ID, Source_Country;

Member_ID,Member_Name,Source_Country,Enrollment_Date,Enrollment_Date_Standard,Last_Flight_Date,Last_Flight_Date_Standard,DOB_Standard
1,Mike,AUS,2022-05-11 00:00:00,2022-05-11T00:00:00.000Z,2022-08-01 00:00:00,2022-08-01T00:00:00.000Z,null
1,Vikas,IND,2022-01-01 00:00:00,2022-01-01T00:00:00.000Z,2022-06-15 00:00:00,2022-06-15T00:00:00.000Z,1998-12-01T00:00:00.000Z
1,Sam,USA,6152022,null,8202022,null,null
2,Jonnathan,AUS,2021-13-13,null,2022-01-05 00:00:00,2022-01-05T00:00:00.000Z,1997-12-13T00:00:00.000Z
2,Rahul,IND,2022-03-05 00:00:00,2022-03-05T00:00:00.000Z,2022-03-10 00:00:00,2022-03-10T00:00:00.000Z,1982-08-13T00:00:00.000Z
2,John,USA,1052022,null,1152022,null,null
3,Cristina,AUS,2022-03-12 00:00:00,2022-03-12T00:00:00.000Z,2022-03-20 00:00:00,2022-03-20T00:00:00.000Z,1998-03-12T00:00:00.000Z
3,Sameer,IND,2022-02-20 00:00:00,2022-02-20T00:00:00.000Z,2022-02-25 00:00:00,2022-02-25T00:00:00.000Z,1952-08-13T00:00:00.000Z
3,Mike,USA,12282021,2021-12-28T00:00:00.000Z,12302021,2021-12-30T00:00:00.000Z,null


In [0]:
--Fixing USA Date

-- Step 6.3: Handle inconsistent USA date formats

CREATE OR REPLACE TEMP VIEW member_dates AS

SELECT
    *,

    COALESCE(
        TRY_TO_TIMESTAMP(Enrollment_Date, 'yyyy-MM-dd HH:mm:ss'),
        TRY_TO_TIMESTAMP(
            CASE
                WHEN Source_Country = 'USA'
                THEN LPAD(TRIM(Enrollment_Date), 8, '0')
                ELSE Enrollment_Date
            END,
            'MMddyyyy'
        )
    ) AS Enrollment_Date_Standard,

    COALESCE(
        TRY_TO_TIMESTAMP(Last_Flight_Date, 'yyyy-MM-dd HH:mm:ss'),
        TRY_TO_TIMESTAMP(
            CASE
                WHEN Source_Country = 'USA'
                THEN LPAD(TRIM(Last_Flight_Date), 8, '0')
                ELSE Last_Flight_Date
            END,
            'MMddyyyy'
        )
    ) AS Last_Flight_Date_Standard,

    TRY_TO_TIMESTAMP(DOB, 'yyyy-MM-dd HH:mm:ss') AS DOB_Standard

FROM bronze_members;

In [0]:
SELECT
    Member_ID,
    Member_Name,
    Source_Country,
    Enrollment_Date,
    Enrollment_Date_Standard,
    Last_Flight_Date,
    Last_Flight_Date_Standard
FROM member_dates
ORDER BY Member_ID, Source_Country;

Member_ID,Member_Name,Source_Country,Enrollment_Date,Enrollment_Date_Standard,Last_Flight_Date,Last_Flight_Date_Standard
1,Mike,AUS,2022-05-11 00:00:00,2022-05-11T00:00:00.000Z,2022-08-01 00:00:00,2022-08-01T00:00:00.000Z
1,Vikas,IND,2022-01-01 00:00:00,2022-01-01T00:00:00.000Z,2022-06-15 00:00:00,2022-06-15T00:00:00.000Z
1,Sam,USA,6152022,2022-06-15T00:00:00.000Z,8202022,2022-08-20T00:00:00.000Z
2,Jonnathan,AUS,2021-13-13,null,2022-01-05 00:00:00,2022-01-05T00:00:00.000Z
2,Rahul,IND,2022-03-05 00:00:00,2022-03-05T00:00:00.000Z,2022-03-10 00:00:00,2022-03-10T00:00:00.000Z
2,John,USA,1052022,2022-01-05T00:00:00.000Z,1152022,2022-01-15T00:00:00.000Z
3,Cristina,AUS,2022-03-12 00:00:00,2022-03-12T00:00:00.000Z,2022-03-20 00:00:00,2022-03-20T00:00:00.000Z
3,Sameer,IND,2022-02-20 00:00:00,2022-02-20T00:00:00.000Z,2022-02-25 00:00:00,2022-02-25T00:00:00.000Z
3,Mike,USA,12282021,2021-12-28T00:00:00.000Z,12302021,2021-12-30T00:00:00.000Z


In [0]:
--Date Quality Flags

/*MISSING: The original source value is empty or NULL.
INVALID: A value exists, but it cannot be parsed.
VALID: The date was parsed successfully.*/

CREATE OR REPLACE TEMP VIEW member_date_quality AS

SELECT
    *,

    CASE
        WHEN Enrollment_Date IS NULL
             OR TRIM(Enrollment_Date) = ''
            THEN 'MISSING'
        WHEN Enrollment_Date_Standard IS NULL
            THEN 'INVALID'
        ELSE 'VALID'
    END AS Enrollment_Date_Status,

    CASE
        WHEN Last_Flight_Date IS NULL
             OR TRIM(Last_Flight_Date) = ''
            THEN 'MISSING'
        WHEN Last_Flight_Date_Standard IS NULL
            THEN 'INVALID'
        ELSE 'VALID'
    END AS Last_Flight_Date_Status,

    CASE
        WHEN DOB IS NULL OR TRIM(DOB) = ''
            THEN 'MISSING'
        WHEN DOB_Standard IS NULL
            THEN 'INVALID'
        ELSE 'VALID'
    END AS DOB_Status

FROM member_dates;

In [0]:
SELECT
    Member_ID,
    Member_Name,
    Source_Country,
    Enrollment_Date_Status,
    Last_Flight_Date_Status,
    DOB_Status
FROM member_date_quality
ORDER BY Member_ID, Source_Country;

Member_ID,Member_Name,Source_Country,Enrollment_Date_Status,Last_Flight_Date_Status,DOB_Status
1,Mike,AUS,VALID,VALID,INVALID
1,Vikas,IND,VALID,VALID,VALID
1,Sam,USA,VALID,VALID,MISSING
2,Jonnathan,AUS,INVALID,VALID,VALID
2,Rahul,IND,VALID,VALID,VALID
2,John,USA,VALID,VALID,MISSING
3,Cristina,AUS,VALID,VALID,VALID
3,Sameer,IND,VALID,VALID,VALID
3,Mike,USA,VALID,VALID,MISSING


In [0]:
--calculating age and stale member

-- Step 6.5: Calculate Age and Stale Member

CREATE OR REPLACE TEMP VIEW member_enriched AS

SELECT
    *,

    CASE
        WHEN DOB_Status = 'VALID'
        THEN FLOOR(
            MONTHS_BETWEEN(CURRENT_DATE(), TO_DATE(DOB_Standard)) / 12
        )
        ELSE NULL
    END AS Age,

    CASE
        WHEN Last_Flight_Date_Status != 'VALID'
            THEN NULL
        WHEN DATEDIFF(
            CURRENT_DATE(),
            TO_DATE(Last_Flight_Date_Standard)
        ) > 90
            THEN TRUE
        ELSE FALSE
    END AS Stale_Member

FROM member_date_quality;

In [0]:
SELECT
    Member_ID,
    Member_Name,
    Source_Country,
    DOB_Status,
    Age,
    Last_Flight_Date_Standard,
    Stale_Member
FROM member_enriched
ORDER BY Member_ID, Source_Country;

Member_ID,Member_Name,Source_Country,DOB_Status,Age,Last_Flight_Date_Standard,Stale_Member
1,Mike,AUS,INVALID,null,2022-08-01T00:00:00.000Z,true
1,Vikas,IND,VALID,27,2022-06-15T00:00:00.000Z,true
1,Sam,USA,MISSING,null,2022-08-20T00:00:00.000Z,true
2,Jonnathan,AUS,VALID,28,2022-01-05T00:00:00.000Z,true
2,Rahul,IND,VALID,44,2022-03-10T00:00:00.000Z,true
2,John,USA,MISSING,null,2022-01-15T00:00:00.000Z,true
3,Cristina,AUS,VALID,28,2022-03-20T00:00:00.000Z,true
3,Sameer,IND,VALID,74,2022-02-25T00:00:00.000Z,true
3,Mike,USA,MISSING,null,2021-12-30T00:00:00.000Z,true


In [0]:
--Builduing silver data table



CREATE OR REPLACE TABLE delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
USING DELTA
AS
SELECT
    Member_Name,
    Member_ID,

    TO_DATE(Enrollment_Date_Standard) AS Enrollment_Date,
    TO_DATE(Last_Flight_Date_Standard) AS Last_Flight_Date,
    TO_DATE(DOB_Standard) AS DOB,

    Tier_Code,
    Agent_Name,
    State,
    Country,
    Post_Code,
    Is_Active,
    Member_Type,
    Source_Country,

    Age,
    Stale_Member,

    Enrollment_Date_Status,
    Last_Flight_Date_Status,
    DOB_Status,

    Ingested_At

FROM member_enriched;

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`;

Member_Name,Member_ID,Enrollment_Date,Last_Flight_Date,DOB,Tier_Code,Agent_Name,State,Country,Post_Code,Is_Active,Member_Type,Source_Country,Age,Stale_Member,Enrollment_Date_Status,Last_Flight_Date_Status,DOB_Status,Ingested_At
Mike,1,2022-05-11,2022-08-01,null,PLT,null,null,null,null,null,null,AUS,null,true,VALID,VALID,INVALID,2026-11-21T07:18:39.239Z
Jonnathan,2,null,2022-01-05,1997-12-13,GLD,null,null,null,null,null,null,AUS,28,true,INVALID,VALID,VALID,2026-11-21T07:18:39.239Z
Cristina,3,2022-03-12,2022-03-20,1998-03-12,SLV,null,null,null,null,null,null,AUS,28,true,VALID,VALID,VALID,2026-11-21T07:18:39.239Z
Vikas,1,2022-01-01,2022-06-15,1998-12-01,SLV,null,null,null,null,null,I,IND,27,true,VALID,VALID,VALID,2026-11-21T07:18:39.239Z
Rahul,2,2022-03-05,2022-03-10,1982-08-13,GLD,null,null,null,null,null,C,IND,44,true,VALID,VALID,VALID,2026-11-21T07:18:39.239Z
Sameer,3,2022-02-20,2022-02-25,1952-08-13,GLD,null,null,null,null,null,I,IND,74,true,VALID,VALID,VALID,2026-11-21T07:18:39.239Z
Sam,1,2022-06-15,2022-08-20,null,PLT,null,null,null,null,null,null,USA,null,true,VALID,VALID,MISSING,2026-11-21T07:18:39.239Z
John,2,2022-01-05,2022-01-15,null,SLV,null,null,null,null,null,null,USA,null,true,VALID,VALID,MISSING,2026-11-21T07:18:39.239Z
Mike,3,2021-12-28,2021-12-30,null,GLD,null,null,null,null,null,null,USA,null,true,VALID,VALID,MISSING,2026-11-21T07:18:39.239Z


In [0]:
--COuntry wise record distribution


SELECT
    Source_Country,
    COUNT(*) AS Total_Members,
    COUNT(DISTINCT Member_ID) AS Distinct_Member_IDs,
    COUNT_IF(Enrollment_Date_Status = 'INVALID') AS Invalid_Enrollment_Dates
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
GROUP BY Source_Country
ORDER BY Source_Country;

Source_Country,Total_Members,Distinct_Member_IDs,Invalid_Enrollment_Dates
AUS,3,3,1
IND,3,3,0
USA,3,3,0


In [0]:
--  Create country-wise views

CREATE OR REPLACE TEMP VIEW silver_aus AS
SELECT *
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
WHERE Source_Country = 'AUS';

CREATE OR REPLACE TEMP VIEW silver_ind AS
SELECT *
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
WHERE Source_Country = 'IND';

CREATE OR REPLACE TEMP VIEW silver_usa AS
SELECT *
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
WHERE Source_Country = 'USA';

In [0]:
SELECT 'AUS' AS Country, COUNT(*) AS Record_Count FROM silver_aus
UNION ALL
SELECT 'IND', COUNT(*) FROM silver_ind
UNION ALL
SELECT 'USA', COUNT(*) FROM silver_usa;

Country,Record_Count
AUS,3
IND,3
USA,3


In [0]:
--Investigate cross-country Member IDs

-- Step 8.1: Investigate cross-country Member IDs

SELECT
    Member_ID,
    Member_Name,
    Source_Country,
    Enrollment_Date,
    Last_Flight_Date,
    Ingested_At
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
ORDER BY Member_ID, Enrollment_Date DESC;

Member_ID,Member_Name,Source_Country,Enrollment_Date,Last_Flight_Date,Ingested_At
1,Sam,USA,2022-06-15,2022-08-20,2026-11-21T07:18:39.239Z
1,Mike,AUS,2022-05-11,2022-08-01,2026-11-21T07:18:39.239Z
1,Vikas,IND,2022-01-01,2022-06-15,2026-11-21T07:18:39.239Z
2,Rahul,IND,2022-03-05,2022-03-10,2026-11-21T07:18:39.239Z
2,John,USA,2022-01-05,2022-01-15,2026-11-21T07:18:39.239Z
2,Jonnathan,AUS,null,2022-01-05,2026-11-21T07:18:39.239Z
3,Cristina,AUS,2022-03-12,2022-03-20,2026-11-21T07:18:39.239Z
3,Sameer,IND,2022-02-20,2022-02-25,2026-11-21T07:18:39.239Z
3,Mike,USA,2021-12-28,2021-12-30,2026-11-21T07:18:39.239Z


In [0]:
--Validating unique names



SELECT
    Member_Name,
    COUNT(DISTINCT Source_Country) AS Country_Count,
    COLLECT_SET(Source_Country) AS Countries,
    COLLECT_SET(Member_ID) AS Member_IDs
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
GROUP BY Member_Name
HAVING COUNT(DISTINCT Source_Country) > 1;

Member_Name,Country_Count,Countries,Member_IDs
Mike,2,"List(AUS, USA)","List(1, 3)"


In [0]:
-- Investigate member identity combinations

SELECT
    Member_Name,
    Member_ID,
    COUNT(*) AS Record_Count,
    COLLECT_SET(Source_Country) AS Source_Countries,
    MIN(Enrollment_Date) AS First_Enrollment,
    MAX(Enrollment_Date) AS Latest_Enrollment
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
GROUP BY Member_Name, Member_ID
HAVING COUNT(*) > 1;

Member_Name,Member_ID,Record_Count,Source_Countries,First_Enrollment,Latest_Enrollment


In [0]:
-- Latest Record Wins

CREATE OR REPLACE TEMP VIEW latest_member_records AS

SELECT
    *,
    ROW_NUMBER() OVER (
        PARTITION BY Member_Name
        ORDER BY Enrollment_Date DESC NULLS LAST
    ) AS Record_Rank

FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`;

In [0]:
SELECT
    Member_Name,
    Member_ID,
    Source_Country,
    Enrollment_Date,
    Record_Rank
FROM latest_member_records
ORDER BY Member_Name, Record_Rank;

Member_Name,Member_ID,Source_Country,Enrollment_Date,Record_Rank
Cristina,3,AUS,2022-03-12,1
John,2,USA,2022-01-05,1
Jonnathan,2,AUS,null,1
Mike,1,AUS,2022-05-11,1
Mike,3,USA,2021-12-28,2
Rahul,2,IND,2022-03-05,1
Sam,1,USA,2022-06-15,1
Sameer,3,IND,2022-02-20,1
Vikas,1,IND,2022-01-01,1


In [0]:
-- Keep latest record for each member

CREATE OR REPLACE TEMP VIEW latest_members AS

SELECT *
FROM latest_member_records
WHERE Record_Rank = 1;

In [0]:
SELECT
    Member_Name,
    Member_ID,
    Source_Country,
    Enrollment_Date,
    Record_Rank
FROM latest_members
ORDER BY Source_Country, Member_Name;

Member_Name,Member_ID,Source_Country,Enrollment_Date,Record_Rank
Cristina,3,AUS,2022-03-12,1
Jonnathan,2,AUS,null,1
Mike,1,AUS,2022-05-11,1
Rahul,2,IND,2022-03-05,1
Sameer,3,IND,2022-02-20,1
Vikas,1,IND,2022-01-01,1
John,2,USA,2022-01-05,1
Sam,1,USA,2022-06-15,1
